# ⚡ MEGA & MediaFire to Google Drive — 1-Click All-in-One Web GUI
This notebook runs completely in a **single click**. Click the Play (▶) button on the cell below to connect Google Drive, verify engines, and launch the Web GUI automatically!

### 🚀 Features Included:
- **1-Click Startup**: No multiple steps or commands—Drive connection, engine installs, and Web GUI boot together in one go.
- **Auto-Merge (-m)**: Prevents folder downloads from freezing on directory prompts.
- **🛑 Stop Button on GUI**: Cancel or abort active downloads at any time with a single click.
- **Multi-Cloud Support**: Paste both **MEGA** (`mega.nz`) and **MediaFire** (`mediafire.com`) links.
- **Live In-Table Percentage**: Real-time progress (% and speed) displayed directly in the Queue Table.
- **Movie & Subfolder Organization**: Organize multiple files/parts into custom folders in your Google Drive.
- **Public Mobile/PC URL**: Accessible directly in Colab or via a `.gradio.live` link.

In [ ]:
#@title 🚀 1-Click Launch: MEGA & MediaFire to Google Drive Station { display-mode: "form" }
# Click the Play (▶) button on this single cell to start everything automatically!
SHARE_URL = True #@param {type:"boolean"}

import os
import sys
import re
import time
import shutil
import subprocess

# -----------------------------------------------------
# STEP 1 (AUTOMATED): Connect Drive & Setup Engines
# -----------------------------------------------------
print("🔄 [1/3] Connecting Google Drive...")
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"Notice: {e}")

print("⚡ [2/3] Checking & Installing acceleration engines (megacmd + aria2 + GUI)...")
need_apt = False
if not os.path.exists("/usr/bin/mega-cmd") and not os.path.exists("/usr/local/bin/mega-cmd"):
    need_apt = True

if need_apt:
    subprocess.run(["wget", "-q", "-nc", "https://mega.nz/linux/repo/xUbuntu_22.04/amd64/megacmd-xUbuntu_22.04_amd64.deb"], check=False)
    subprocess.run(["sudo", "apt-get", "install", "./megacmd-xUbuntu_22.04_amd64.deb", "aria2", "-y"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    import gradio as gr
    import pandas as pd
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio", "pandas", "beautifulsoup4", "requests"], check=False)
    import gradio as gr
    import pandas as pd
    import requests
    from bs4 import BeautifulSoup

try:
    import pty
    import fcntl
    HAS_PTY = True
except ImportError:
    HAS_PTY = False

print("🚀 [3/3] Launching Full Web Control Center...")

# -----------------------------------------------------
# STEP 2: Application Logic, Auto-Merge & Reset Controls
# -----------------------------------------------------
if 'queue_state' not in globals():
    queue_state = []

# Clean any leftover stuck states from previous browser reloads
for item in queue_state:
    if "Transferring" in item.get("status", ""):
        item["status"] = "Pending ⏳"

is_stopped = False
active_process = None

def clean_mega_daemons():
    """Kills any background daemons cleanly with strict timeouts so it NEVER hangs."""
    try:
        subprocess.run(["mega-quit"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=2)
    except:
        pass
    try:
        subprocess.run(["pkill", "-9", "-f", "mega-cmd"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=2)
        subprocess.run(["pkill", "-9", "-f", "mega-get"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=2)
    except:
        pass
    time.sleep(0.3)

def detect_and_clean_link(link):
    link = link.strip().rstrip('.,;')
    if not link:
        return None, None
    if not link.startswith('http://') and not link.startswith('https://'):
        link = 'https://' + link
        
    if 'mega.nz' in link or 'mega.io' in link or 'mega.co.nz' in link:
        return 'MEGA 🔴', link
    elif 'mediafire.com' in link:
        return 'MediaFire 🔵', link
    elif any(link.lower().endswith(ext) for ext in ['.mp4', '.mkv', '.avi', '.zip', '.rar', '.7z', '.tar', '.gz', '.iso', '.pdf', '.bin']):
        return 'Direct URL 🌐', link
    return 'Link 🌐', link

def extract_links(text):
    if not text:
        return []
    candidates = re.findall(r'https?://[^\s,"\'<>]+', text)
    if not candidates:
        candidates = [line.strip() for line in re.split(r'[\r\n,]+', text) if line.strip()]
    
    items = []
    for c in candidates:
        service, clean = detect_and_clean_link(c)
        if clean and not any(item['link'] == clean for item in items):
            items.append({'service': service, 'link': clean})
    return items

def resolve_mediafire_url(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5'
    }
    res = requests.get(url, headers=headers, allow_redirects=True, timeout=15)
    if res.status_code != 200:
        raise Exception(f"MediaFire returned HTTP status {res.status_code}")
    
    soup = BeautifulSoup(res.text, 'html.parser')
    btn = soup.find('a', id='downloadButton')
    if btn and btn.get('href') and btn.get('href').startswith('http'):
        return btn.get('href')
        
    m = re.findall(r'href=["\'](https?://download\d*\.mediafire\.com/[^"\']+)["\']', res.text)
    if m:
        return m[0]
        
    m2 = re.findall(r'["\'](https?://download\d*\.mediafire\.com/[^"\']+)["\']', res.text)
    if m2:
        return m2[0]
        
    raise Exception("Direct download link not found on MediaFire page (file may be deleted or protected).")

def clean_terminal_output(raw_text):
    lines = []
    for line in raw_text.split('\n'):
        if '\r' in line:
            segs = [s.strip() for s in line.split('\r') if s.strip()]
            if segs:
                lines.append(segs[-1])
        else:
            if line.strip():
                lines.append(line)
    return '\n'.join(lines[-40:])

def get_queue_df():
    if not queue_state:
        return pd.DataFrame(columns=["#", "Source", "Movie / Subfolder", "Link / URL", "Status"])
    rows = []
    for item in queue_state:
        sub = item["subfolder"] if item["subfolder"] else "(Root Folder)"
        rows.append({
            "#": item["id"],
            "Source": item.get("service", "Link"),
            "Movie / Subfolder": sub,
            "Link / URL": item["link"],
            "Status": item["status"]
        })
    return pd.DataFrame(rows)

def add_to_queue(subfolder, links_text):
    global queue_state
    if not links_text or not links_text.strip():
        return get_queue_df(), "", "⚠️ Please paste a link."
        
    if os.path.isfile(links_text.strip()):
        try:
            with open(links_text.strip(), 'r', encoding='utf-8', errors='ignore') as f:
                links_text = f.read()
        except Exception as e:
            return get_queue_df(), "", f"❌ File error: {e}"
            
    clean_sub = re.sub(r'[<>:"/\\|?*]', '_', subfolder.strip()).strip(' .') if subfolder else ""
    found = extract_links(links_text)
    
    if not found:
        return get_queue_df(), links_text, "❌ No valid MEGA or MediaFire links found."
        
    added = 0
    for item in found:
        if not any(q["link"] == item["link"] for q in queue_state):
            queue_state.append({
                "id": len(queue_state) + 1,
                "service": item["service"],
                "subfolder": clean_sub,
                "link": item["link"],
                "status": "Pending ⏳"
            })
            added += 1
            
    folder_name = f"'{clean_sub}'" if clean_sub else "Root"
    return get_queue_df(), "", f"✅ Added {added} link(s) to {folder_name} (Total in queue: {len(queue_state)})"

def remove_last():
    global queue_state
    if queue_state:
        removed = queue_state.pop()
        msg = f"ℹ️ Removed #{removed['id']}: {removed['link']}"
    else:
        msg = "ℹ️ Queue is already empty."
    return get_queue_df(), msg

def clear_queue():
    global queue_state
    queue_state = []
    return get_queue_df(), "ℹ️ Queue cleared."

def get_storage_info():
    path = "/content/drive/MyDrive"
    if not os.path.exists(path):
        return "⚠️ Google Drive not mounted at /content/drive/MyDrive."
    try:
        total, used, free = shutil.disk_usage(path)
        total_gb = total / (1024 ** 3)
        used_gb = used / (1024 ** 3)
        free_gb = free / (1024 ** 3)
        pct = (used / total) * 100
        return f"📊 **Google Drive Storage:** Free: **{free_gb:.2f} GB** | Used: **{used_gb:.2f} GB** ({pct:.1f}%) | Total: **{total_gb:.2f} GB**"
    except Exception as e:
        return f"⚠️ Storage error: {e}"

def stop_all_transfers():
    global is_stopped, active_process
    is_stopped = True
    if active_process and active_process.poll() is None:
        try:
            active_process.terminate()
            time.sleep(0.3)
            if active_process.poll() is None:
                active_process.kill()
        except:
            pass
    clean_mega_daemons()
    try:
        subprocess.run(["pkill", "-9", "aria2c"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except:
        pass
    return "🛑 Stop signal received! Terminating active transfers..."

def run_transfers(base_folder):
    global queue_state, is_stopped, active_process
    is_stopped = False
    
    if not queue_state:
        yield get_queue_df(), "⚠️ Queue is empty! Add links before starting.", "⚠️ Queue is empty.", gr.update(interactive=True), gr.update(interactive=False)
        return
        
    base_folder = base_folder.strip() or "MEGA_Transfers"
    base_target = f"/content/drive/MyDrive/{base_folder}"
    os.makedirs(base_target, exist_ok=True)
    
    clean_mega_daemons()
    
    raw_terminal_log = f"[INIT] Starting Transfer Session\n[DEST] Google Drive -> {base_folder}\n[COUNT] {len(queue_state)} item(s) in queue\n" + ("=" * 50) + "\n"
    yield get_queue_df(), clean_terminal_output(raw_terminal_log), "⏳ Starting transfers...", gr.update(interactive=False), gr.update(interactive=True)
    
    success_count = 0
    fail_count = 0
    
    local_staging_dir = "/content/temp_downloads"
    os.makedirs(local_staging_dir, exist_ok=True)
    
    for idx, item in enumerate(queue_state, 1):
        if is_stopped:
            raw_terminal_log += f"\n🛑 [CANCELLED] Session stopped by user before starting item {idx}.\n"
            yield get_queue_df(), clean_terminal_output(raw_terminal_log), "🛑 Session Stopped.", gr.update(interactive=True), gr.update(interactive=False)
            break
            
        if "Completed" in item["status"]:
            continue
            
        item["status"] = "Transferring ⚡ 0%"
        sub = item.get("subfolder", "").strip()
        if sub:
            dest = os.path.join(base_target, sub)
            dest_label = f"{base_folder} / {sub}"
        else:
            dest = base_target
            dest_label = base_folder
        os.makedirs(dest, exist_ok=True)
        
        for f_name in os.listdir(local_staging_dir):
            f_path = os.path.join(local_staging_dir, f_name)
            try:
                if os.path.isdir(f_path): shutil.rmtree(f_path)
                else: os.remove(f_path)
            except: pass
            
        service = item.get("service", "")
        url = item["link"]
        
        raw_terminal_log += f"\n>>> [TRANSFER {idx}/{len(queue_state)}] ({service})\nFolder: {dest_label}\nLink: {url}\n" + ("-" * 45) + "\n"
        yield get_queue_df(), clean_terminal_output(raw_terminal_log), f"⏳ Downloading [{idx}/{len(queue_state)}]...", gr.update(interactive=False), gr.update(interactive=True)
        
        if 'MediaFire' in service:
            try:
                raw_terminal_log += "[MediaFire] Resolving direct download link...\n"
                yield get_queue_df(), clean_terminal_output(raw_terminal_log), f"⏳ Resolving MediaFire direct link...", gr.update(interactive=False), gr.update(interactive=True)
                direct_url = resolve_mediafire_url(url)
                raw_terminal_log += f"[MediaFire] Direct link resolved! Launching aria2 16x accelerated download...\n"
                cmd = ["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--auto-file-renaming=false", "--allow-overwrite=true", "--summary-interval=1", direct_url, "-d", local_staging_dir]
            except Exception as e:
                item["status"] = "Failed ❌"
                fail_count += 1
                raw_terminal_log += f"❌ [ERROR] Failed to resolve MediaFire URL: {e}\n"
                yield get_queue_df(), clean_terminal_output(raw_terminal_log), f"Failed {idx}/{len(queue_state)}", gr.update(interactive=False), gr.update(interactive=True)
                continue
        elif 'MEGA' in service:
            # CRITICAL: -m (auto-merge existing folders) prevents mega-get from hanging on interactive merge prompts!
            cmd = ["mega-get", "-m", "--ignore-quota-warn", url, local_staging_dir]
        else:
            cmd = ["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--auto-file-renaming=false", "--allow-overwrite=true", "--summary-interval=1", url, "-d", local_staging_dir]
            
        if HAS_PTY:
            master, slave = pty.openpty()
            process = subprocess.Popen(cmd, stdin=subprocess.DEVNULL, stdout=slave, stderr=slave, text=True, close_fds=True)
            os.close(slave)
            active_process = process
            
            fl = fcntl.fcntl(master, fcntl.F_GETFL)
            fcntl.fcntl(master, fcntl.F_SETFL, fl | os.O_NONBLOCK)
            
            last_yield = time.time()
            last_data_time = time.time()
            
            while True:
                if is_stopped:
                    if process.poll() is None:
                        process.terminate()
                        process.wait()
                    item["status"] = "Stopped 🛑"
                    raw_terminal_log += f"\n🛑 [STOPPED] Transfer {idx} was stopped by user.\n"
                    yield get_queue_df(), clean_terminal_output(raw_terminal_log), "🛑 Transfers Stopped.", gr.update(interactive=True), gr.update(interactive=False)
                    break
                    
                try:
                    chunk = os.read(master, 1024)
                    if chunk:
                        chunk_str = chunk.decode('utf-8', errors='ignore')
                        raw_terminal_log += chunk_str
                        last_data_time = time.time()
                        pct_m = re.findall(r'(\d+(?:\.\d+)?)\s*%', chunk_str)
                        if pct_m:
                            item["status"] = f"Transferring ⚡ {pct_m[-1]}%"
                        if time.time() - last_yield > 0.35:
                            yield get_queue_df(), clean_terminal_output(raw_terminal_log), f"⏳ Downloading [{idx}/{len(queue_state)}] ({item['status']})...", gr.update(interactive=False), gr.update(interactive=True)
                            last_yield = time.time()
                except OSError:
                    pass
                    
                if process.poll() is not None:
                    try:
                        rem = os.read(master, 4096)
                        if rem:
                            rem_str = rem.decode('utf-8', errors='ignore')
                            raw_terminal_log += rem_str
                            pct_m = re.findall(r'(\d+(?:\.\d+)?)\s*%', rem_str)
                            if pct_m:
                                item["status"] = f"Transferring ⚡ {pct_m[-1]}%"
                    except:
                        pass
                    break
                time.sleep(0.1)
                
                # Real-time stall & quota watchdog check
                if time.time() - last_data_time > 25:
                    try:
                        diag = subprocess.run(["mega-transfers"], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, timeout=4)
                        if diag.stdout and ("PAUSED" in diag.stdout or "WAITING" in diag.stdout or "LIMIT" in diag.stdout.upper()):
                            raw_terminal_log += f"\n⚠️ [STATUS ALERT] MEGA Bandwidth Notice:\n{diag.stdout.strip()}\n(If bandwidth limit is reached on this IP, reconnect Colab runtime to get a fresh IP address)\n"
                            yield get_queue_df(), clean_terminal_output(raw_terminal_log), "⚠️ Bandwidth Limit Warning", gr.update(interactive=False), gr.update(interactive=True)
                        elif diag.stdout:
                            raw_terminal_log += f"\n[ACTIVE TRANSFERS]\n{diag.stdout.strip()}\n"
                            yield get_queue_df(), clean_terminal_output(raw_terminal_log), f"⏳ Transferring...", gr.update(interactive=False), gr.update(interactive=True)
                    except:
                        pass
                    last_data_time = time.time()
                    
            try:
                os.close(master)
            except:
                pass
        else:
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            active_process = process
            for line in process.stdout:
                if is_stopped:
                    process.terminate()
                    item["status"] = "Stopped 🛑"
                    break
                raw_terminal_log += line
                pct_m = re.findall(r'(\d+(?:\.\d+)?)\s*%', line)
                if pct_m:
                    item["status"] = f"Transferring ⚡ {pct_m[-1]}%"
                yield get_queue_df(), clean_terminal_output(raw_terminal_log), f"⏳ Downloading [{idx}/{len(queue_state)}]...", gr.update(interactive=False), gr.update(interactive=True)
            process.wait()
            
        if is_stopped:
            break
            
        if process.poll() is None:
            process.terminate()
            process.wait()
            
        if process.returncode == 0:
            raw_terminal_log += f"\n📦 [SYNC] Moving downloaded files to Google Drive ({dest_label})...\n"
            yield get_queue_df(), clean_terminal_output(raw_terminal_log), f"📦 Syncing to Google Drive...", gr.update(interactive=False), gr.update(interactive=True)
            try:
                for item_name in os.listdir(local_staging_dir):
                    s_item = os.path.join(local_staging_dir, item_name)
                    d_item = os.path.join(dest, item_name)
                    if os.path.exists(d_item):
                        if os.path.isdir(d_item): shutil.rmtree(d_item)
                        else: os.remove(d_item)
                    shutil.move(s_item, dest)
                item["status"] = "Completed ✅ 100%"
                success_count += 1
                raw_terminal_log += f"✅ [DONE] Transferred & Synced into '{dest_label}' successfully!\n"
            except Exception as move_err:
                item["status"] = "Failed ❌"
                fail_count += 1
                raw_terminal_log += f"❌ [SYNC ERROR] Failed to move to Google Drive: {move_err}\n"
        else:
            item["status"] = "Failed ❌"
            fail_count += 1
            raw_terminal_log += f"\n❌ [FAILED] Process exited with code {process.returncode}\n"
            
        yield get_queue_df(), clean_terminal_output(raw_terminal_log), f"Completed {idx}/{len(queue_state)}", gr.update(interactive=False), gr.update(interactive=True)
        
    active_process = None
    clean_mega_daemons()
    
    if is_stopped:
        raw_terminal_log += "\n" + ("=" * 50) + f"\n🛑 Session Stopped by user.\n"
        yield get_queue_df(), clean_terminal_output(raw_terminal_log), "🛑 Transfers Stopped.", gr.update(interactive=True), gr.update(interactive=False)
    else:
        raw_terminal_log += "\n" + ("=" * 50) + f"\n🎉 Transfers Finished: {success_count} Succeeded | {fail_count} Failed\n"
        summary_msg = f"🎉 Finished! {success_count}/{len(queue_state)} succeeded."
        if fail_count > 0:
            summary_msg += f" ({fail_count} failed)"
        yield get_queue_df(), clean_terminal_output(raw_terminal_log), summary_msg, gr.update(interactive=True), gr.update(interactive=False)

with gr.Blocks(title="Cloud Transfer Station (MEGA & MediaFire)", css=custom_css, theme=gr.themes.Default()) as demo:
    gr.HTML("""
    <div class="header-box">
        <div class="header-title">⚡ MEGA & MediaFire to Google Drive Station</div>
        <div class="header-sub">High-Speed Cloud Direct Transfer with Multi-Cloud Queue & Folder Grouping</div>
    </div>
    """)
    
    with gr.Tabs():
        with gr.TabItem("🚀 Transfers Dashboard"):
            with gr.Row():
                with gr.Column(scale=4):
                    gr.Markdown("### 📁 1. Destination Folder")
                    base_folder_input = gr.Textbox(
                        label="Base Google Drive Folder",
                        value="MEGA_Transfers",
                        placeholder="e.g. MEGA_Transfers"
                    )
                    subfolder_input = gr.Textbox(
                        label="Movie / Subfolder Name (Optional)",
                        placeholder="e.g. Amaran or Movie (2024)",
                        info="All links added will save inside this subfolder"
                    )
                    
                    gr.Markdown("### 🔗 2. Add MEGA or MediaFire Links")
                    links_input = gr.Textbox(
                        label="Links (MEGA or MediaFire)",
                        placeholder="Paste MEGA (mega.nz/...) or MediaFire (mediafire.com/...) links here...",
                        lines=3
                    )
                    
                    with gr.Row():
                        add_btn = gr.Button("➕ Add to Queue", elem_id="add_btn", scale=2)
                        remove_btn = gr.Button("➖ Remove Last", scale=1)
                        clear_btn = gr.Button("🗑️ Clear", scale=1)
                    
                    status_notice = gr.Markdown("💡 *Ready. Paste your MEGA or MediaFire links above.*")
                    
                    # Transfer Action Controls: Start and Stop
                    with gr.Row():
                        start_btn = gr.Button("🚀 Start All Transfers", elem_id="start_btn", scale=3, size="lg")
                        stop_btn = gr.Button("🛑 Stop", elem_id="stop_btn", scale=1, size="lg", interactive=False)
                
                with gr.Column(scale=6):
                    gr.Markdown("### 📋 Current Transfer Queue")
                    queue_table = gr.Dataframe(
                        value=get_queue_df(),
                        headers=["#", "Source", "Movie / Subfolder", "Link / URL", "Status"],
                        datatype=["number", "str", "str", "str", "str"],
                        interactive=False,
                        wrap=True
                    )
                    
                    gr.Markdown("### 💻 Real-Time Terminal Progress")
                    log_box = gr.Textbox(
                        label="Console Stream",
                        value="Waiting for transfer session to start...\n",
                        lines=14,
                        max_lines=18,
                        autoscroll=True,
                        interactive=False,
                        elem_classes=["terminal-card"]
                    )
                    
        with gr.TabItem("📊 Google Drive Tools & Storage"):
            with gr.Column():
                storage_box = gr.Markdown(value=get_storage_info())
                refresh_storage_btn = gr.Button("🔄 Refresh Storage Usage", size="sm")
                
                gr.Markdown("---")
                gr.Markdown("### 🛠️ Engine Reset & Diagnostics")
                gr.Markdown("If MEGA ever pauses or a daemon hangs, click here to instantly purge all background workers.")
                reset_status = gr.Markdown("")
                reset_btn = gr.Button("🧹 Reset Engine & Clear Stuck Queues")
                
                def reset_action():
                    clean_mega_daemons()
                    return "✅ All background daemons and stuck jobs cleared successfully!"
                
                reset_btn.click(reset_action, outputs=[reset_status])
                
                gr.Markdown("---")
                gr.Markdown("### ⚡ Google Drive Cache Sync")
                gr.Markdown("Force flushes Google Drive write cache so files appear immediately on your Google Drive mobile app and browser.")
                sync_status = gr.Markdown("")
                flush_btn = gr.Button("⚡ Force Flush & Sync Drive")
                
                def flush_action():
                    try:
                        from google.colab import drive
                        drive.flush_and_unmount()
                        return "✅ Google Drive cache flushed and unmounted successfully!"
                    except Exception as e:
                        return f"ℹ️ Notice: {e}"
                        
                flush_btn.click(flush_action, outputs=[sync_status])
                refresh_storage_btn.click(get_storage_info, outputs=[storage_box])
                
    add_btn.click(
        fn=add_to_queue,
        inputs=[subfolder_input, links_input],
        outputs=[queue_table, links_input, status_notice]
    )
    
    remove_btn.click(
        fn=remove_last,
        outputs=[queue_table, status_notice]
    )
    
    clear_btn.click(
        fn=clear_queue,
        outputs=[queue_table, status_notice]
    )
    
    start_event = start_btn.click(
        fn=run_transfers,
        inputs=[base_folder_input],
        outputs=[queue_table, log_box, status_notice, start_btn, stop_btn]
    )
    
    stop_btn.click(
        fn=stop_all_transfers,
        inputs=None,
        outputs=[status_notice],
        cancels=[start_event]
    )

print("\n\033[94m[INFO] Starting Gradio Web Server...\033[0m")
demo.queue(default_concurrency_limit=10).launch(share=SHARE_URL, debug=False)
